# Import

In [ ]:
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

from torchvision import datasets, transforms

import math

# Configuration

In [ ]:
class Config:
    lr = 3e-4
    device = "cuda" if torch.cuda.is_available() else "cpu"
    beta_min = 0.1
    beta_max = 20
    img_chans = 1
    img_size = 32
    epochs = 20
    batch_size = 128

CFG = Config()

print(CFG.device)

cuda


# Data Load

In [ ]:
transform = transforms.Compose([
                transforms.Resize((32, 32)),
                transforms.ToTensor(),
                transforms.Normalize((0.5,), (0.5,))
            ])

mnist_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
cifar10_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)

mnist_dataloader = DataLoader(mnist_dataset, batch_size=CFG.batch_size, shuffle=True)
cifar10_dataloader = DataLoader(cifar10_dataset, batch_size=CFG.batch_size, shuffle=True)

100%|██████████| 9.91M/9.91M [00:00<00:00, 18.2MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 500kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.81MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 7.10MB/s]
100%|██████████| 170M/170M [00:03<00:00, 46.3MB/s]


# Define SDE

In [ ]:
class SDE:
    def sde(self, x, t):
        raise NotImplementedError

    def marginal_prob(self, x0, t):
        raise NotImplementedError

    def prior_sampling(self, shape):
        raise NotImplementedError

class VP_SDE(SDE):
    def __init__(self, beta, beta_integral):
        self.beta = beta
        self.beta_integral = beta_integral

    def sde(self, x, t):
        beta = self.beta(t)[:, None, None, None].to(CFG.device)
        drift = -0.5 * beta * x
        diffusion = beta ** 0.5
        return drift, diffusion

    def marginal_prob(self, x0, t):
        alpha_bar = torch.exp(-self.beta_integral(t))[:, None, None, None].to(CFG.device)
        mean = x0 * (alpha_bar ** 0.5)
        std = (1 - alpha_bar) ** 0.5
        return mean.to(CFG.device), std.to(CFG.device)

    def prior_sampling(self, shape):
        return torch.randn(shape).to(CFG.device)


beta = lambda t: CFG.beta_min + (CFG.beta_max - CFG.beta_min) * t
beta_integral = lambda t: CFG.beta_min * t + (CFG.beta_max - CFG.beta_min) * t ** 2 / 2

vp_sde = VP_SDE(beta, beta_integral)

# Define Model

In [ ]:
class ScoreFunction(nn.Module):
    def __init__(self, model, sde):
        super().__init__()
        self.model = model
        self.sde = sde

    def forward(self, x, t, c):
        epsilon = self.model(x, t, c)
        _, std = self.sde.marginal_prob(torch.zeros_like(x), t)
        return - epsilon / std

class UNet(nn.Module):
    def __init__(self, in_channels=1, base_channels=64, ratios=(1,2,3,4), num_classes=10, dropout=0.1):
        super().__init__()
        chans = [base_channels * ratio for ratio in ratios]

        self.time_embed = TimeEmbedding(emb_dim=4*base_channels)
        self.cond_embed = nn.Embedding(num_classes, embedding_dim=4*base_channels)

        self.stem = nn.Conv2d(in_channels, base_channels, 3, padding=1)

        self.down = nn.ModuleList([
            DownStage(chans[0], chans[1]),
            DownStage(chans[1], chans[2]),
            DownStage(chans[2], chans[3], attn=True),
        ])
        self.middle = MidBlock(chans[3])
        self.up = nn.ModuleList([
            UpStage(chans[3] + chans[2], chans[2], attn=True),
            UpStage(chans[2] + chans[1], chans[1]),
            UpStage(chans[1] + chans[0], chans[0]),
        ])
        self.head = nn.Sequential(
            nn.GroupNorm(8, chans[0]),
            nn.SiLU(),
            nn.Conv2d(chans[0], in_channels, 3, padding=1)
        )

    def forward(self, x, t, cond=None):
        temb = self.time_embed(t)
        cemb = self.cond_embed(cond) if not cond is None else 0
        emb = temb + cemb

        h = self.stem(x)
        skips = []

        for stage in self.down:
            h, stage_skips = stage(h, emb)
            skips.append(stage_skips)

        h = self.middle(h, emb)

        for i, stage in enumerate(self.up, 1):
            h = stage(h, skips[-i], emb)

        y = self.head(h)
        return y

class DownStage(nn.Module):
    def __init__(self, in_channels, out_channels, attn=False):
        super().__init__()
        self.attn = attn
        self.resblock1 = ResBlock(in_channels, in_channels)
        self.resblock2 = ResBlock(in_channels, in_channels)
        if attn:
            self.attention = AttentionBlock(in_channels)
        self.downsample = nn.Conv2d(in_channels, out_channels, 3, stride=2, padding=1)

    def forward(self, x, emb):
        x = self.resblock1(x, emb)
        x = self.resblock2(x, emb)
        if self.attn:
            x = self.attention(x)
        x_down = self.downsample(x)
        return x_down, x

class UpStage(nn.Module):
    def __init__(self, in_channels, out_channels, attn=False):
        super().__init__()
        self.attn = attn
        self.resblock1 = ResBlock(in_channels, out_channels)
        self.resblock2 = ResBlock(out_channels, out_channels)
        if attn:
            self.attention = AttentionBlock(out_channels)

    def forward(self, x, skip, emb):
        B, C, H, W = x.shape
        x = nn.functional.interpolate(x, (H * 2, W * 2), mode='nearest')
        x = torch.concat((x, skip), dim=1)
        x = self.resblock1(x, emb)
        x = self.resblock2(x, emb)
        if self.attn:
            x = self.attention(x)
        return x

class MidBlock(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.resblock1 = ResBlock(in_channels, in_channels)
        self.resblock2 = ResBlock(in_channels, in_channels)

    def forward(self, x, emb):
        x = self.resblock1(x, emb)
        x = self.resblock2(x, emb)
        return x

class ResBlock(nn.Module):
    def __init__(self, in_channels, out_channels, dropout=0.1):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels

        self.nonlinearity = nn.SiLU()
        self.normalize1 = nn.GroupNorm(8, in_channels)
        self.normalize2 = AdaGN(out_channels)
        self.dropout = nn.Dropout(dropout)
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)

        if in_channels != out_channels:
            self.shortcut = nn.Conv2d(in_channels, out_channels, 1)

    def forward(self, x, emb):
        if self.in_channels != self.out_channels:
            shortcut = self.shortcut(x)
        else:
            shortcut = x

        x = self.normalize1(x)
        x = self.nonlinearity(x)
        x = self.conv1(x)
        x = self.normalize2(x, emb)
        x = self.nonlinearity(x)
        x = self.dropout(x)
        x = self.conv2(x)
        return shortcut + x

class AdaGN(nn.Module):
    def __init__(self, in_channels, emb_dim=256):
        super().__init__()
        self.in_channels = in_channels
        self.normalize = nn.GroupNorm(8, in_channels, affine=False)
        self.dense = nn.Linear(emb_dim, in_channels * 2)

    def forward(self, x, emb):
        x = self.normalize(x)
        scale, shift = torch.split(self.dense(emb), self.in_channels, dim=1)
        x = x * (1 + scale[:, :, None, None]) + shift[:, :, None, None]
        return x

class AttentionBlock(nn.Module):
    def __init__(self, in_channels, num_heads=2):
        super().__init__()
        self.normalize = nn.GroupNorm(num_groups=8, num_channels=in_channels)
        self.attention = nn.MultiheadAttention(embed_dim=in_channels, num_heads=num_heads, batch_first=True)

    def forward(self, x):
        h = self.normalize(x)
        B, C, H, W = h.shape
        tokens = h.view(B, C, H*W).transpose(1, 2)
        y, _ = self.attention(tokens, tokens, tokens)
        y = y.transpose(1, 2).view(B, C, H, W)
        return x + y

class TimeEmbedding(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.emb_dim = emb_dim
        self.mlp = nn.Sequential(
            nn.Linear(emb_dim, emb_dim),
            nn.SiLU(),
            nn.Linear(emb_dim, emb_dim)
        )

    def forward(self, t):
        half = self.emb_dim // 2
        freqs = torch.exp(
            -math.log(10000) * torch.arange(half, device=t.device) / (half - 1)
        )
        args = t[:, None] * freqs[None, :]
        time_emb = torch.cat([torch.sin(args), torch.cos(args)], dim=-1)
        time_emb = self.mlp(time_emb)
        return time_emb

score_theta = ScoreFunction(UNet().to(CFG.device), vp_sde)
optimizer = torch.optim.AdamW(score_theta.model.parameters(), lr=CFG.lr)

# Define Loss

In [ ]:
class LossFn(nn.Module):
    def __init__(self, sde):
        super().__init__()
        self.sde = sde

    def forward(self, x0, c, score_fn):
        t = torch.rand(x0.size(0), device=CFG.device)
        e = torch.randn_like(x0, device=CFG.device)
        mean, std = self.sde.marginal_prob(x0, t)
        x_t = mean + std * e
        e_hat = score_fn.model(x_t.to(CFG.device), t, c)
        loss = torch.mean((e - e_hat) ** 2)
        return loss

loss_fn = LossFn(vp_sde)

# Define Sampler

In [ ]:
class DDPMSampler:
    def __init__(self, steps = 1000):
        self.steps = steps

    @torch.inference_mode()
    def sample(self, score_fn, sde, nums, cond):
        shape = (nums, CFG.img_chans, CFG.img_size, CFG.img_size)
        x = sde.prior_sampling(shape)
        t_grid = torch.linspace(1, 0, self.steps + 1)
        dt = 1 / self.steps
        for i in range(self.steps):
            t = t_grid[i].expand(shape[0]).to(CFG.device)
            f, g = sde.sde(x, t)
            score = score_fn(x, t, cond)
            drift = f - g ** 2 * score
            x -= drift * dt + g * torch.randn_like(x) * dt ** 0.5
        return x

class DDIMSampler:
    def __init__(self, steps = 100):
        self.steps = steps

    @torch.inference_mode()
    def sample(self, score_fn, sde, nums, cond):
        shape = (nums, CFG.img_chans, CFG.img_size, CFG.img_size)
        x = sde.prior_sampling(shape)
        t_grid = torch.linspace(1, 0, self.steps + 1)
        dt = 1 / self.steps
        for i in range(self.steps):
            t = t_grid[i].expand(shape[0]).to(CFG.device)
            f, g = sde.sde(x, t)
            score = score_fn(x, t, cond)
            drift = f - 0.5 * g ** 2 * score
            x -= drift * dt
        return x


#class DPMSolver:

ddpm_sampler = DDPMSampler()
ddim_sampler = DDIMSampler(200)
#dpm_solver = DPMSolver()

# Training

In [ ]:
for epoch in range(CFG.epochs):
    print(f'##### EPOCH {epoch + 1} #####')
    running_loss = 0.0
    for i, (x0, cond) in enumerate(mnist_dataloader):
        score_theta.model.train()

        x0 = x0.to(CFG.device)
        cond = cond.to(CFG.device)

        loss = loss_fn(x0, cond, score_theta)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        if (i + 1) % 100 == 0:
            avg_loss = running_loss / 100
            print(f"[Step {i+1}/{len(mnist_dataloader)} | running loss: {avg_loss:.6f}")
            running_loss = 0.0  # 다음 구간을 위해 초기화

##### EPOCH 1 #####
[Step 100/469 | running loss: 0.177289
[Step 200/469 | running loss: 0.049275
[Step 300/469 | running loss: 0.039349
[Step 400/469 | running loss: 0.034286
##### EPOCH 2 #####
[Step 100/469 | running loss: 0.031283
[Step 200/469 | running loss: 0.029285
[Step 300/469 | running loss: 0.027229
[Step 400/469 | running loss: 0.027326
##### EPOCH 3 #####
[Step 100/469 | running loss: 0.025012
[Step 200/469 | running loss: 0.023436
[Step 300/469 | running loss: 0.023454
[Step 400/469 | running loss: 0.023786
##### EPOCH 4 #####
[Step 100/469 | running loss: 0.022315
[Step 200/469 | running loss: 0.021603
[Step 300/469 | running loss: 0.021583
[Step 400/469 | running loss: 0.020923
##### EPOCH 5 #####
[Step 100/469 | running loss: 0.020567
[Step 200/469 | running loss: 0.020144
[Step 300/469 | running loss: 0.020112
[Step 400/469 | running loss: 0.020223
##### EPOCH 6 #####
[Step 100/469 | running loss: 0.019561
[Step 200/469 | running loss: 0.018899
[Step 300/469 | runnin

# Sampling

In [ ]:
n_samples = 16
cond = torch.ones((n_samples,), dtype=torch.long, device=CFG.device) * 5

score_theta.model.eval()
images = ddim_sampler.sample(score_theta, vp_sde, n_samples, cond)


In [ ]:
import matplotlib.pyplot as plt

plt.imshow(images[16].permute(1, 2, 0).cpu().detach().numpy())
plt.show()

IndexError: index 16 is out of bounds for dimension 0 with size 16

In [ ]:
def print_params_by_layer(model: nn.Module) -> None:
    """
    '직접' 파라미터를 갖는 모듈(=leaf module) 단위로 파라미터 수를 집계해 출력합니다.
    """
    # 각 파라미터의 "소유 모듈" 이름을 매핑
    owner = {}
    for mod_name, mod in model.named_modules():
        for p_name, p in mod.named_parameters(recurse=False):
            full = f"{mod_name}.{p_name}" if mod_name else p_name
            owner[full] = mod_name if mod_name else "<root>"

    # leaf 모듈만 출력(하위 모듈을 가진 모듈은 요약에 중복될 수 있어서 제외)
    leaf_names = {name for name, m in model.named_modules() if len(list(m.children())) == 0}

    rows = []
    for mod_name, mod in model.named_modules():
        if mod_name not in leaf_names:
            continue
        total = sum(p.numel() for p in mod.parameters(recurse=False))
        trainable = sum(p.numel() for p in mod.parameters(recurse=False) if p.requires_grad)
        if total == 0:
            continue
        rows.append((mod_name if mod_name else "<root>", mod.__class__.__name__, total, trainable))

    rows.sort(key=lambda x: x[2], reverse=True)

    # 출력
    header = f"{'module':40s} {'type':20s} {'params':>12s} {'trainable':>12s}"
    print(header)
    print("-" * len(header))
    for mod_name, typ, total, trainable in rows:
        print(f"{mod_name:40s} {typ:20s} {total:12d} {trainable:12d}")

    total_all = sum(p.numel() for p in model.parameters())
    train_all = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print("-" * len(header))
    print(f"{'TOTAL':40s} {'':20s} {total_all:12d} {train_all:12d}")